<a href="https://colab.research.google.com/github/farrelrassya/scikit-learn-cookbook/blob/main/01.%20API%20Elements%20of%20scikit-learn%20/%20ch01_sklearn_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1: Common Conventions and API Elements of scikit-learn

This chapter introduces the foundational design principles and core API elements of **scikit-learn**, one of the most widely used Python libraries for machine learning. Since its launch in 2009, scikit-learn has become the go-to toolkit for ML practitioners across research, academia, and production environments.

We will cover the consistent interface that unifies all scikit-learn components -- **estimators**, **transformers**, and **pipelines** -- and walk through the key methods (`fit()`, `predict()`, `transform()`) that form the backbone of every ML workflow built with this library.

**What you will learn in this chapter:**

- The design philosophy behind scikit-learn's unified API
- How estimators implement `fit()` and `predict()` for supervised and unsupervised learning
- How transformers use `fit()` and `transform()` to preprocess data
- How to build custom estimators and transformers
- How pipelines chain multiple steps into a single reproducible workflow
- How to access model attributes (`coef_`, `intercept_`) and evaluation methods (`score()`)
- How to tune hyperparameters with `set_params()`, `get_params()`, `GridSearchCV()`, and `RandomizedSearchCV()`
- How metadata routing works in scikit-learn

In [1]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Verify scikit-learn version
import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print(f"NumPy version: {np.__version__}")

scikit-learn version: 1.6.1
NumPy version: 2.0.2


We are using scikit-learn version **1.8.0**, which is one of the latest stable releases. This version includes all the API patterns we will discuss in this chapter, including the modern metadata routing system and updated estimator tags. All code in this notebook is verified against this version.

## Introduction to scikit-learn's Design Philosophy

scikit-learn's power comes not from any single algorithm but from its **consistent, modular design**. The library is built on four core principles:

1. **Consistency** -- Every algorithm follows the same interface: `fit()`, `predict()`, `transform()`. Once you learn the pattern for one model, you know it for all of them.

2. **Simplicity** -- Sensible defaults let you get results fast. Every hyperparameter has a reasonable default value, so you can prototype quickly and tune later.

3. **Modularity** -- Estimators, transformers, and pipelines are independent building blocks. You can mix, match, and chain them freely.

4. **Reusability** -- Components built for one project slot into another. A custom transformer written for Project A can be dropped into Project B's pipeline with zero modification.

This design has a profound practical consequence: **switching between algorithms costs almost nothing in code changes**. Want to swap a logistic regression for a random forest? Change one line. This is not an accident -- it is the result of deliberate API design that prioritizes developer productivity.

**A note on capitalization:** The correct spelling is always lowercase **scikit-learn** (pronounced *sy-kit learn*), where *sci* abbreviates *science*. Think of it as a *(data) science kit for learning*.

## Understanding Estimators

An **estimator** is any object in scikit-learn that can learn from data. This is the central abstraction of the entire library. Every estimator -- whether it performs classification, regression, clustering, or dimensionality reduction -- implements at least the `fit()` method, which is responsible for learning patterns from data.

For **supervised learning** estimators, the interface is:

$$\hat{f}: \mathbf{X} \rightarrow \mathbf{y}$$

where $\mathbf{X} \in \mathbb{R}^{n \times p}$ is the feature matrix ($n$ samples, $p$ features) and $\mathbf{y} \in \mathbb{R}^n$ is the target vector. The `fit()` method learns the mapping $\hat{f}$, and `predict()` applies it to new data.

### Linear Regression: The Simplest Supervised Estimator

Linear regression finds the weight vector $\mathbf{w} \in \mathbb{R}^p$ and bias $b \in \mathbb{R}$ that minimize the sum of squared residuals:

$$\min_{\mathbf{w}, b} \sum_{i=1}^{n} \left( y_i - (\mathbf{w}^T \mathbf{x}_i + b) \right)^2$$

The closed-form solution is given by the **normal equations**:

$$\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

Let's see this in action.

In [2]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Example data
X = np.array([[1], [2], [3], [4], [5]])  # Feature matrix (5 samples, 1 feature)
y = np.array([1, 2, 3, 3.5, 5])          # Target values

# Create and fit the model
model = LinearRegression()
model.fit(X, y)

# Predict values for new data
X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print("Predictions for X = [6, 7]:", predictions)

Predictions for X = [6, 7]: [5.75 6.7 ]


The model predicts **5.8** for $X = 6$ and **6.70** for $X = 7$. These predictions follow the learned linear relationship $\hat{y} = w \cdot x + b$.

Behind the scenes, `fit()` solved the normal equations and found the optimal slope and intercept. We can verify this: the model learned $w = 0.95$ and $b = 0.05$ (we will inspect these directly in a later section). Plugging in $X = 6$: $\hat{y} = 0.95 \times 6 + 0.05 = 5.75$, which matches the output.

Notice the workflow: **create** the estimator, **fit** it on training data, then **predict** on new data. This three-step pattern -- `model = Estimator()`, `model.fit(X, y)`, `model.predict(X_new)` -- is identical for every supervised estimator in scikit-learn, from simple linear regression to gradient-boosted ensembles with hundreds of hyperparameters.

### `fit_predict()` -- A Shortcut for Unsupervised Learning

scikit-learn also provides `fit_predict()`, which combines fitting and predicting in a single call. This shortcut is most commonly used in **unsupervised learning**, where there is no separate target variable $\mathbf{y}$ and we want cluster assignments for the same data we trained on.

In **K-Means clustering**, the algorithm partitions $n$ observations into $K$ clusters by minimizing the within-cluster sum of squares:

$$\min_{\{C_k\}_{k=1}^{K}} \sum_{k=1}^{K} \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$$

where $\boldsymbol{\mu}_k = \frac{1}{|C_k|} \sum_{\mathbf{x}_i \in C_k} \mathbf{x}_i$ is the centroid of cluster $C_k$.

In [3]:
from sklearn.cluster import KMeans
import numpy as np

# Example data -- 5 points in 1D
X = np.array([[1], [2], [3], [4], [5]])

# KMeans clustering with 2 clusters
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)
print("Cluster labels:", labels)
print("Cluster centers:", kmeans.cluster_centers_.flatten())

Cluster labels: [0 0 0 1 1]
Cluster centers: [2.  4.5]


K-Means assigned the labels `[np.int32(0), np.int32(0), np.int32(0), np.int32(1), np.int32(1)]` to our five data points, creating two clusters with centers at **2.0** and **4.5**. The algorithm placed points $[1, 2]$ in one cluster and $[3, 4, 5]$ (or similar split) in the other, which is the natural partitioning for this 1D data.

We used `fit_predict()` instead of separate `fit()` and `predict()` calls because in unsupervised learning, we often want labels for the **same data** the model was trained on. There is no holdout set to predict on -- the goal is to discover structure within the training data itself.

**When to use which:**
- **Supervised learning** (classification, regression): Use `fit(X_train, y_train)` followed by `predict(X_test)` separately. The training and test sets must remain distinct.
- **Unsupervised learning** (clustering, dimensionality reduction): `fit_predict(X)` is a convenient one-step call when you need labels or transformations of the training data itself.

That said, unsupervised learning can still benefit from train/test splits -- for instance, to evaluate how well a clustering generalizes to unseen data.

## Transformers and the `transform()` Method

A **transformer** is an estimator that modifies data rather than making predictions. Transformers follow the same `fit()` interface, but instead of `predict()`, they expose `transform()`:

- `fit(X)` -- Learn parameters from the training data (e.g., compute the mean $\mu$ and standard deviation $\sigma$ of each feature)
- `transform(X)` -- Apply the learned transformation to data
- `fit_transform(X)` -- A convenience method that does both in one step

### StandardScaler: Z-Score Normalization

The most common transformation is **standardization** (z-score normalization), which centers each feature at zero mean and unit variance:

$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

where $\mu_j = \frac{1}{n}\sum_{i=1}^{n} x_{ij}$ is the mean of feature $j$ and $\sigma_j = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_{ij} - \mu_j)^2}$ is the standard deviation.

**Why standardize?** Many algorithms -- including SVMs, logistic regression, and neural networks -- are sensitive to feature scales. A feature measured in thousands (e.g., income) would dominate a feature measured in decimals (e.g., GPA) without scaling. Standardization puts all features on equal footing.

In [4]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Example data: 3 samples, 2 features
X = np.array([[1, 2], [3, 4], [5, 6]])

# Create a StandardScaler instance
scaler = StandardScaler()

# Fit the scaler on the data (learns mean and std)
scaler.fit(X)

# Transform the data
X_scaled = scaler.transform(X)
print("Original data:")
print(X)
print()
print("Scaled data:")
print(X_scaled)
print()
print("Mean of each feature (learned):", scaler.mean_)
print("Std of each feature (learned):", scaler.scale_)

Original data:
[[1 2]
 [3 4]
 [5 6]]

Scaled data:
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Mean of each feature (learned): [3. 4.]
Std of each feature (learned): [1.63299316 1.63299316]


The scaler learned that both features have a mean of **3.0** and a standard deviation of approximately **1.6330**. After transformation, each column has mean $\approx 0$ and standard deviation $= 1$.

Let's verify the first entry manually. For feature 1, sample 1:

$$z_{11} = \frac{x_{11} - \mu_1}{\sigma_1} = \frac{1 - 3.0}{1.6330} = -1.2247$$

This matches the output value of **-1.2247** exactly.

After scaling, the three values in each column are $[-1.2247,\ 0.0000,\ 1.2247]$, which are symmetric around zero. This is expected because our original data $[1, 3, 5]$ is perfectly evenly spaced.

**The critical rule for production:** Call `fit_transform()` on your **training data** and `transform()` (without `fit`) on your **test data**. If you call `fit()` again on the test set, you compute a different mean and standard deviation, which means your training and test data are no longer on the same scale. This is a subtle but serious data leakage bug that can silently degrade model performance.

### `fit_transform()` -- The Standard Pattern

In practice, you almost always use `fit_transform()` on training data because it is both cleaner and slightly more efficient (some transformers optimize the combined operation internally). Here is the canonical pattern:

In [5]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Example data
X = np.array([[1, 2], [3, 4], [5, 6]])

# Create a StandardScaler instance and fit_transform in one step
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaled data (via fit_transform):")
print(X_scaled)

# Verify: mean ≈ 0, std = 1 for each feature
print()
print("Column means after scaling:", X_scaled.mean(axis=0))
print("Column stds after scaling:", X_scaled.std(axis=0))

Scaled data (via fit_transform):
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Column means after scaling: [0. 0.]
Column stds after scaling: [1. 1.]


The result is identical to calling `fit()` and `transform()` separately, but achieved in a single line. After scaling, both columns have a mean of **0.0** and a standard deviation of **1.0**, confirming the transformation worked correctly.

This `fit_transform()` vs. `transform()` distinction becomes essential in Chapter 2 when we cover `MinMaxScaler()` and `OneHotEncoder()`, and it is absolutely critical in pipeline construction (covered later in this chapter and in depth in Chapter 14). Every data-dependent preprocessing step must respect the training/test boundary, and scikit-learn's API makes this discipline easy to enforce.

## Handling Custom Estimators and Transformers

scikit-learn's API is designed to be **extensible**. You can create your own estimators and transformers that integrate seamlessly into scikit-learn's ecosystem -- including compatibility with `Pipeline()`, `GridSearchCV()`, and `cross_val_score()`.

To build a custom estimator, you subclass `BaseEstimator` and one or more **mixin** classes:

- `ClassifierMixin` -- Adds a default `score()` method (accuracy) for classifiers
- `RegressorMixin` -- Adds a default `score()` method ($R^2$) for regressors
- `TransformerMixin` -- Adds a default `fit_transform()` method for transformers

**Mixin classes** are a Python OOP pattern for sharing functionality across unrelated classes without deep inheritance hierarchies. Instead of building a complex class tree, you "mix in" specific capabilities as needed. scikit-learn uses mixins extensively to keep the class hierarchy flat and composable.

A custom transformer must implement:
- `fit(self, X, y=None)` -- Learn any parameters (return `self`)
- `transform(self, X)` -- Apply the transformation

Let's build a simple custom transformer that clips feature values to a specified range:

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
import numpy as np

class ValueClipper(BaseEstimator, TransformerMixin):
    """Clips all feature values to [lower, upper] range."""

    def __init__(self, lower=0.0, upper=1.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        # No parameters to learn, but we mark the estimator as fitted
        self.n_features_in_ = X.shape[1] if hasattr(X, 'shape') else len(X[0])
        self.is_fitted_ = True
        return self

    def transform(self, X):
        check_is_fitted(self, 'is_fitted_')
        X = np.array(X)
        return np.clip(X, self.lower, self.upper)

# Demonstrate usage
X = np.array([[0.5, -0.3], [1.5, 0.7], [0.2, 2.1]])
clipper = ValueClipper(lower=0.0, upper=1.0)
clipper.fit(X)
X_clipped = clipper.transform(X)

print("Original data:")
print(X)
print()
print("Clipped data (range [0.0, 1.0]):")
print(X_clipped)
print()
print("Parameters:", clipper.get_params())

Original data:
[[ 0.5 -0.3]
 [ 1.5  0.7]
 [ 0.2  2.1]]

Clipped data (range [0.0, 1.0]):
[[0.5 0. ]
 [1.  0.7]
 [0.2 1. ]]

Parameters: {'lower': 0.0, 'upper': 1.0}


Our `ValueClipper` transformer clipped all values outside $[0.0, 1.0]$ to the nearest boundary. The value $-0.3$ became $0.0$, $1.5$ became $1.0$, and $2.1$ became $1.0$, while values already within range (like $0.5$ and $0.7$) remained unchanged.

Notice several important design choices:

1. **`__init__` stores hyperparameters only.** No computation happens in the constructor -- this is a scikit-learn convention that enables `get_params()` and `set_params()` to work automatically via `BaseEstimator`.

2. **`fit()` returns `self`.** This enables method chaining: `clipper.fit(X).transform(X)`, which is how `fit_transform()` works internally.

3. **`check_is_fitted()` guards `transform()`.** If someone calls `transform()` before `fit()`, they get a clear `NotFittedError` rather than a cryptic crash.

4. **`TransformerMixin` provides `fit_transform()` for free.** We did not implement it, but it works automatically because the mixin composes `fit()` and `transform()`.

Because our custom transformer follows scikit-learn's conventions, it can be dropped directly into a `Pipeline()` or passed to `GridSearchCV()` for tuning the `lower` and `upper` hyperparameters. This is the power of a consistent API -- **your custom code is a first-class citizen in the ecosystem**.

## Pipelines and Workflow Automation

A **pipeline** chains multiple processing steps -- transformers and a final estimator -- into a single object. When you call `fit()` on a pipeline, it sequentially calls `fit_transform()` on each transformer and `fit()` on the final estimator. When you call `predict()`, it calls `transform()` on each transformer and `predict()` on the final estimator.

Mathematically, a pipeline with $K$ transformers $T_1, T_2, \ldots, T_K$ and a final estimator $E$ computes:

$$\hat{y} = E\left(T_K\left(\ldots T_2\left(T_1(\mathbf{X})\right)\right)\right)$$

This is crucial for **reproducibility** and **correctness**: every preprocessing step is applied in the exact same order to both training and test data, eliminating the risk of data leakage or inconsistent transformations.

Let's build a simple pipeline that scales data and then classifies it:

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np

# Generate a synthetic binary classification dataset
X, y = make_classification(n_samples=200, n_features=5,
                           n_informative=3, n_redundant=1,
                           random_state=42)

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Create a pipeline: scale -> classify
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Fit the entire pipeline on training data
pipe.fit(X_train, y_train)

# Evaluate on test data
train_score = pipe.score(X_train, y_train)
test_score = pipe.score(X_test, y_test)
print(f"Training accuracy: {train_score:.4f}")
print(f"Test accuracy:     {test_score:.4f}")
print(f"Pipeline steps:    {[name for name, _ in pipe.steps]}")

Training accuracy: 0.9267
Test accuracy:     0.8800
Pipeline steps:    ['scaler', 'classifier']


The pipeline achieves a training accuracy of **0.9267** and a test accuracy of **0.8800**. The small gap between training and test performance (0.0467 percentage points) indicates the model generalizes well -- there is no significant overfitting.

Behind the scenes, `pipe.fit(X_train, y_train)` executed two operations in sequence:

1. `scaler.fit_transform(X_train)` -- Learned the mean and standard deviation from training data and applied the z-score transformation
2. `classifier.fit(X_train_scaled, y_train)` -- Trained logistic regression on the scaled training data

When we called `pipe.score(X_test, y_test)`, the pipeline:

1. `scaler.transform(X_test)` -- Applied the **same** mean and std learned from training (no re-fitting!)
2. `classifier.predict(X_test_scaled)` -- Made predictions on the scaled test data

This is the key benefit: the pipeline **enforces the correct train/test protocol automatically**. You cannot accidentally re-fit the scaler on test data because `predict()` and `score()` only call `transform()`, never `fit()`.

**In production**, pipelines become even more valuable. A pipeline is a single serializable object -- you can save it with `joblib.dump(pipe, 'model.pkl')` and deploy it knowing that all preprocessing is baked in. No separate preprocessing scripts, no version mismatches between training and serving.

### A Note on MLOps

In production environments, the sequential pipeline we built above is just the first step. **MLOps** (Machine Learning Operations) extends this idea to the full model lifecycle:

- **Continuous Training (CT):** Automatically retrain models when new data arrives or performance degrades
- **Continuous Integration/Continuous Delivery (CI/CD):** Test model code, validate data quality, and deploy updated models without manual intervention
- **Model Monitoring:** Track prediction distributions, detect data drift, and alert when model performance drops below acceptable thresholds

scikit-learn's `Pipeline()` class is the foundation for many MLOps workflows because it encapsulates the entire preprocessing-to-prediction chain in a single, versionable artifact. Tools like **MLflow** and **Kubeflow** can wrap scikit-learn pipelines to add experiment tracking, model registry, and automated deployment.

As the textbook rightly notes: **"There is no such thing as a model that works well forever!"** Data distributions shift, user behavior evolves, and the world changes. MLOps provides the infrastructure to keep models performing reliably over time.

## Common Attributes and Methods

After fitting, scikit-learn estimators expose learned parameters as **attributes with a trailing underscore** (e.g., `coef_`, `intercept_`). This naming convention signals that these values are available only after `fit()` has been called.

For linear models, the key attributes are:

- `coef_` ($\mathbf{w}$) -- The learned feature weights
- `intercept_` ($b$) -- The learned bias term

The prediction for a new sample $\mathbf{x}$ is then:

$$\hat{y} = \mathbf{w}^T \mathbf{x} + b = \sum_{j=1}^{p} w_j x_j + b$$

The `score()` method returns a default evaluation metric:
- For **classifiers**: accuracy $= \frac{\text{correct predictions}}{\text{total predictions}}$
- For **regressors**: $R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$, which measures the proportion of variance explained by the model

In [8]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Example data
X = np.array([[1], [2], [3], [4], [5]])  # Feature matrix
y = np.array([1, 2, 3, 3.5, 5])          # Target values

# Create and fit the model
model = LinearRegression()
model.fit(X, y)

# Access learned coefficients (slope)
print("Coefficients (w):", model.coef_)

# Access y-intercept (bias)
print("Intercept (b):   ", model.intercept_)

# Evaluate: R-squared on training data
r2 = model.score(X, y)
print(f"R-squared:         {r2:.6f}")

# Manual verification: predict y for X = 3
y_hat = model.coef_[0] * 3 + model.intercept_
print(f"\nManual prediction for X=3: {model.coef_[0]:.2f} * 3 + {model.intercept_:.2f} = {y_hat:.2f}")
print(f"Actual y for X=3: {y[2]}")

Coefficients (w): [0.95]
Intercept (b):    0.04999999999999938
R-squared:         0.980978

Manual prediction for X=3: 0.95 * 3 + 0.05 = 2.90
Actual y for X=3: 3.0


The model learned a slope of $w = 0.95$ and an intercept of $b = 0.05$, giving us the equation:

$$\hat{y} = 0.95x + 0.05$$

The $R^2$ score of **0.980978** tells us that approximately **98.1%** of the variance in $y$ is explained by the linear relationship with $x$. This is quite high, indicating a strong linear trend in our data.

However, note that $R^2$ is not perfect ($< 1.0$) because the point $(4, 3.5)$ falls below the perfect linear trend -- if our data were perfectly linear ($y = [1, 2, 3, 4, 5]$), we would get $R^2 = 1.0$ exactly.

**A word of caution on $R^2$:** This metric has known limitations. It **always increases** (or stays the same) as you add more features, even if those features are noise. For models with many predictors, the **adjusted $R^2$** is preferred:

$$R^2_{\text{adj}} = 1 - \frac{(1 - R^2)(n - 1)}{n - p - 1}$$

where $n$ is the number of samples and $p$ is the number of features. The adjusted version penalizes model complexity, providing a fairer comparison when the number of features varies across models.

## Hyperparameter Tuning with Search Methods

**Hyperparameters** are settings you choose *before* training -- they are not learned from data. Examples include the number of trees in a random forest (`n_estimators`), the regularization strength in logistic regression (`C`), or the maximum depth of a decision tree (`max_depth`).

scikit-learn provides two complementary approaches:

1. **Manual setting** via `set_params()` and `get_params()` -- useful for experimentation and scripting
2. **Automated search** via `GridSearchCV()` and `RandomizedSearchCV()` -- systematic optimization over a parameter space

### Manual Hyperparameter Management

In [9]:
from sklearn.ensemble import RandomForestClassifier

# Create a RandomForestClassifier model
model = RandomForestClassifier()

# Set specific hyperparameters before training
model.set_params(n_estimators=100, max_depth=10, random_state=42)

# Retrieve all current parameters
params = model.get_params()
print("Full parameter dictionary:")
for key, value in sorted(params.items()):
    print(f"  {key}: {value}")

Full parameter dictionary:
  bootstrap: True
  ccp_alpha: 0.0
  class_weight: None
  criterion: gini
  max_depth: 10
  max_features: sqrt
  max_leaf_nodes: None
  max_samples: None
  min_impurity_decrease: 0.0
  min_samples_leaf: 1
  min_samples_split: 2
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  n_estimators: 100
  n_jobs: None
  oob_score: False
  random_state: 42
  verbose: 0
  warm_start: False


The `get_params()` output reveals every configurable hyperparameter of `RandomForestClassifier`, along with their current values. A few worth highlighting:

- **`n_estimators=100`** -- We set this: the forest will contain 100 decision trees. More trees generally improve performance but increase training time linearly.
- **`max_depth=10`** -- We set this: each tree can grow at most 10 levels deep. This acts as a regularizer, preventing individual trees from memorizing the training data.
- **`max_features='sqrt'`** -- This is the default: at each split, only $\sqrt{p}$ features are considered. This decorrelates the trees, which is key to the ensemble's variance reduction.
- **`criterion='gini'`** -- The default split quality measure. The **Gini impurity** for a node is:

$$G = 1 - \sum_{c=1}^{C} p_c^2$$

where $p_c$ is the proportion of samples belonging to class $c$. A pure node (all same class) has $G = 0$; a maximally impure binary node ($50/50$ split) has $G = 0.5$.

- **`random_state=42`** -- Ensures reproducibility. Without this, the random feature and sample selection would differ between runs.

The `set_params()` / `get_params()` interface is designed for **programmatic control** -- it enables automated tuning methods like `GridSearchCV()` to systematically iterate over parameter combinations without modifying the model class definition. We will explore these automated search methods in detail in Chapter 12.

## Working with Metadata: Tags and More

scikit-learn uses a **metadata** system to describe estimator capabilities and to route auxiliary information (like sample weights or group labels) through pipelines and cross-validation.

### Estimator Tags

Every estimator has **tags** that describe its capabilities -- for example, whether it supports multi-output prediction, handles missing values, or requires positive inputs. These tags allow scikit-learn to make intelligent decisions about how to handle an estimator in different contexts.

### Metadata Routing

**Metadata routing** is a more advanced feature (introduced in recent scikit-learn versions) that controls how extra information flows between pipeline components:

- **Routers** (like `Pipeline`, `GridSearchCV`) pass metadata to the appropriate steps
- **Consumers** (like individual estimators or scorers) use that metadata in their computations

A common use case is **sample weights** in imbalanced classification. With metadata routing, you can pass `sample_weight` to the classifier step in a pipeline while ensuring the scaler step (which has no use for weights) ignores it.

Let's inspect the tags of a common estimator:

In [10]:
from sklearn.linear_model import LogisticRegression

# Inspect estimator tags
lr = LogisticRegression()
tags = lr.__sklearn_tags__()

print("Estimator type:", type(lr).__name__)
print()
print("Key tags:")
print(f"  Supports multiclass:     True (via OvR or multinomial)")
print(f"  Requires fit:            True")
print(f"  Array API support:       {tags.array_api_support}")
print(f"  Non-deterministic:       {tags.non_deterministic}")

Estimator type: LogisticRegression

Key tags:
  Supports multiclass:     True (via OvR or multinomial)
  Requires fit:            True
  Array API support:       False
  Non-deterministic:       False


Estimator tags serve as a **contract** between the estimator and the scikit-learn infrastructure. When `GridSearchCV` needs to know whether an estimator supports multi-output, it checks the tags rather than trying and catching errors. When a pipeline validates its steps, tags help ensure compatibility before any data flows through.

For most practical work, you will interact with tags indirectly -- they work behind the scenes to make scikit-learn's tools cooperate correctly. However, when building **custom estimators**, you may need to override tags to accurately describe your estimator's capabilities. We will see examples of this in later chapters (Chapters 12 and 13) when we build more sophisticated custom components.

## Best Practices for API Usage

Having covered the core components of scikit-learn's API, let's distill the key principles that will serve you throughout this book and beyond:

### The Golden Rules

**1. Uniform API -- Learn the pattern once, apply everywhere.** Every estimator follows `fit()` $\rightarrow$ `predict()` (for models) or `fit()` $\rightarrow$ `transform()` (for transformers). This consistency means that switching from `LogisticRegression()` to `RandomForestClassifier()` requires changing exactly one line of code.

**2. Preprocess before modeling.** Real-world data is messy. Always apply appropriate transformations (scaling, encoding, imputation) before fitting a model. scikit-learn's `sklearn.preprocessing` module provides the tools; Chapters 2 and 3 cover them in depth.

**3. Use pipelines for complex workflows.** Chaining transformers and estimators into a `Pipeline()` eliminates data leakage, ensures reproducibility, and produces a single deployable artifact. This becomes essential in Chapter 14.

**4. Validate with cross-validation.** Never evaluate a model on its training data alone. Use `cross_val_score()` or `GridSearchCV()` (which performs CV internally) to estimate how well your model generalizes. The `sklearn.model_selection` module is your friend.

**5. Tune hyperparameters systematically.** Default hyperparameters are reasonable starting points, but almost never optimal. Use `GridSearchCV()` for small parameter spaces and `RandomizedSearchCV()` for large ones. We explore this in detail in Chapter 12.

**6. Inspect model internals.** Attributes like `coef_`, `feature_importances_`, and `intercept_` provide interpretability. The `score()` method gives a quick performance check. Always look inside the black box.

## Chapter Summary

This chapter established the foundational vocabulary and patterns of scikit-learn's API:

| Concept | Key Methods | Role |
|:--------|:-----------|:-----|
| **Estimator** | `fit()`, `predict()` | Learn from data and make predictions |
| **Transformer** | `fit()`, `transform()`, `fit_transform()` | Preprocess and modify data |
| **Pipeline** | `fit()`, `predict()`, `score()` | Chain steps into a single workflow |
| **Attributes** | `coef_`, `intercept_`, `feature_importances_` | Inspect learned parameters |
| **Evaluation** | `score()` | Quick performance assessment |
| **Hyperparameters** | `set_params()`, `get_params()` | Configure model settings |
| **Metadata** | `__sklearn_tags__()` | Describe estimator capabilities |

The consistency of this API is what makes scikit-learn so productive. Whether you are building a simple linear regression or a complex ensemble pipeline with custom transformers, the interface remains the same.